# {{场景}}指标验证 — 每个数都能自己核

每个指标分别说明：口径、粒度、公式、中间量、完整结果与对账。

- 复制时替换字段、窗口、容差、读取函数和计算占位；目标接口需先检查或补齐实现。
- 只读第三步归档的 processed，不重放清洗链；源数据标准有问题回第三步。
- 完整明细保留命名 dataframe，输出格只放全量汇总或字段说明。
- 逐项业务确认后沉淀稳定计算；正式入口同次运行产生全部结果表及 JSON。

In [ ]:
import pandas as pd
from read_data import read_processed_procurement_inbound_meta

# 示例来源；替换为实际数据集的显式 processed 读取。缺档/过期应在读取时失败。
clean, read_meta = read_processed_procurement_inbound_meta("{{起始周期}}", "{{结束周期}}")
AMOUNT_COLUMN = "{{金额列}}"
AMOUNT_ATOL = float("{{业务确认的金额绝对容差}}")
SCENE = "{{场景}}"
BUSINESS_END = "{{业务结束日}}"
parameters = {"{{参数名}}": "{{参数值}}"}  # 替换为实际可复跑参数

pd.DataFrame([{
    "行数": len(clean),
    "金额合计": pd.to_numeric(clean[AMOUNT_COLUMN], errors="raise").sum(min_count=1),
    "实际覆盖": read_meta["coverage"],
}])

## 二、{{指标名}}

**口径**：{{一句话大白话，说什么算什么}}　**粒度**：{{一行 = 一个物料×一个月}}，加总 = {{什么}}

In [ ]:
# 示例中间量；替换为当前指标已确认的聚合，不在这里标准化来源字段。
mid = (
    clean.groupby(["{{维度列}}", "{{期间列}}"], dropna=False, as_index=False)
    .agg(数量=("{{量列}}", "sum"))
)
print(f"完整中间量变量 mid，共 {len(mid):,} 行")
# 如 mid 是明细，仅保留变量；输出另做全量分组摘要。

In [ ]:
def calculate_confirmed_metric(processed_rows, intermediate):
    raise NotImplementedError("填写方案1.4确认的公式，返回完整指标结果 dataframe")

result = calculate_confirmed_metric(clean, mid)
print(f"完整指标结果变量 result，共 {len(result):,} 行")

In [ ]:
def build_reconciliation(processed_rows, metric_rows):
    # 参照数来自已知报表或有定位条件的手算；本表值从实际计算结果提取。
    # 每行包含 对什么、参照数、本表、容差、定位条件；覆盖三件套。
    raise NotImplementedError("填写实际参照数与计算结果的核对，不手填通过结论")

reconciliation = build_reconciliation(clean, result)
reconciliation["差异"] = reconciliation["本表"] - reconciliation["参照数"]
reconciliation["一致"] = (
    reconciliation["参照数"].notna() & reconciliation["本表"].notna()
    & reconciliation["差异"].abs().le(reconciliation["容差"])
)
assert not reconciliation.empty and reconciliation["一致"].all(), "对账不通过，先定位差异"
reconciliation

### 这张报表长这样（未来 `data/outputs/` 里的那张表）

一行 = {{一个物料×一个月}}，加总 = {{...}}。列、单位、粒度有不合适的，就在这格说，改了重跑本节。

In [ ]:
# 完整报表候选，按本节确认的粒度与列调整，供 Data Wrangler 翻查。
report = result.copy()
print(f"完整报表变量 report，共 {len(report):,} 行")
columns = pd.DataFrame({"列": report.columns, "类型": report.dtypes.astype(str).to_numpy()})
columns["口径/单位"] = "{{逐列填写}}"
columns

**你的结论**：{{数字可信 / 口径要改（回 1.4，记 4.2）}} —— 定了再看下一个指标

---

{{「一节」照此复制，一个指标一份，编号往后排}}

## 最后、和稳定计算对拍（逐表确认后才跑）

`confirmed_tables` 收录本 notebook 已确认的完整结果表；稳定分析模块使用同一窗口、参数和输入版本。
逐张按声明的键对齐，比字段、空值和指标值；不能只看汇总金额相等。

In [ ]:
confirmed_tables = {"detail": report}  # 月度/分组等已确认表也放进来，并分别填写键及各指标容差
TABLE_KEYS = {"detail": ["{{主键列}}"]}
COLUMN_ATOL = {"detail": {AMOUNT_COLUMN: AMOUNT_ATOL}}  # 其他指标需要容差时按单位明确填写

def run_confirmed_calculation(processed_rows, parameters):
    raise NotImplementedError("调用真实稳定分析函数，返回命名结果表 dict")

script_tables = run_confirmed_calculation(clean, parameters)
assert set(script_tables) == set(confirmed_tables), "结果表集合不一致"
for name, expected in confirmed_tables.items():
    actual = script_tables[name]
    keys = TABLE_KEYS[name]
    assert not expected.duplicated(keys).any() and not actual.duplicated(keys).any(), f"{name} 粒度不唯一"
    expected = expected.sort_values(keys).reset_index(drop=True)
    actual = actual.sort_values(keys).reset_index(drop=True)
    assert list(actual.columns) == list(expected.columns), f"{name} 字段不一致"
    for column in expected.columns:
        pd.testing.assert_series_equal(
            actual[column], expected[column], check_dtype=True,
            check_exact=False, rtol=0, atol=COLUMN_ATOL[name].get(column, 0),
        )
print(f"稳定计算与 notebook 的 {len(script_tables)} 张结果表对拍通过")

**待确认事项**：{{逐条注明是否阻塞落地}}；口径变更记入 4.2。

本 notebook 保留为核对基准；更换数据时按实际输入版本重跑，不能只复用旧输出格。

## 输出表定稿（每张确认过的表记一张卡，抄进方案 4.3）

表清单 = 下面这几张在 notebook 里被业务点头的报表，不再另起一套字段名。

In [ ]:
pd.DataFrame([
    {"表名": "detail", "粒度(主键)": "{{业务键}}", "列及单位": "{{确认的列与单位口径}}",
     "更新方式": "{{更新方式}}", "看板用途": "{{KPI/趋势/下钻}}",
     "parquet": f"data/outputs/{SCENE}/{BUSINESS_END[:4]}/{BUSINESS_END[:7]}/{BUSINESS_END}/data.parquet"},
    # 每张额外确认表增加一行，路径为同目录 tables/<表名>.parquet，不能只进 JSON。
])

## 归档与指定批次读回

下格示意稳定入口的归档段；正式命令必须在同一次运行调用，不能依赖 notebook 手工补档。
目标 `write_outputs()` 接口需先实现并验证，不能用附表只进 JSON 的旧函数冒充。
所有确认表各有 parquet，并生成同批 `data.json` 和 `manifest.json`；全部校验通过才发布。

In [ ]:
from archive import write_outputs

paths = write_outputs(
    tables=script_tables,
    scene=SCENE,
    snapshot_date=BUSINESS_END,
    meta={
        "inputs": read_meta, "parameters": parameters,
        "window_start": "{{窗口起}}", "window_end": "{{窗口止}}",
        "entrypoint": "{{实际按场景命名的脚本命令}}",
    },
)
paths  # batch_id、parquet(表名→路径)、json、manifest

**逐表读回核对**：使用本次实际批次，不取「最新」；比较表集合、字段、行数、空值和每个指标值。
日期字段按归档声明格式还原；数值按各列确认的容差比较，不能先把缺失金额变成零。

In [ ]:
import json
from read_data import read_outputs

payload = json.loads(paths["json"].read_text(encoding="utf-8"))
manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
assert payload["meta"]["batch_id"] == manifest["batch_id"] == paths["batch_id"]
assert set(payload["tables"]) == set(paths["parquet"]) == set(script_tables)
DATE_COLUMNS = {"detail": []}  # 按表填写已声明的日期列；有日期时不能保留空清单
checks = []
for name, expected in script_tables.items():
    back = read_outputs(SCENE, BUSINESS_END, table=name, batch_id=paths["batch_id"])
    pd.testing.assert_frame_equal(back, expected)  # parquet 对稳定计算：保留字段、类型及空值
    block = payload["tables"][name]
    assert list(back.columns) == block["columns"] and len(back) == len(block["rows"])
    json_rows = pd.DataFrame(block["rows"], columns=block["columns"])
    for column in back.columns:
        left, right = back[column].reset_index(drop=True), json_rows[column].reset_index(drop=True)
        if column in DATE_COLUMNS.get(name, []):
            left, right = pd.to_datetime(left), pd.to_datetime(right)
        pd.testing.assert_series_equal(
            left, right, check_dtype=False, check_exact=False,
            rtol=0, atol=COLUMN_ATOL[name].get(column, 0),
        )
    checks.append({"表名": name, "parquet行数": len(back), "JSON行数": len(json_rows),
                   "状态": "全部字段、空值与指标断言通过"})
pd.DataFrame(checks)

**交付**：将实际命令、参数、输入版本、`paths["batch_id"]`、各表 parquet、JSON 与 manifest 路径记进方案 4.3。
第五步只内嵌这批 `data.json`，不回读 raw/processed，不在 HTML 补算业务指标。
最终 Excel 如需交付，消费同批已计算结果；研究确认阶段不导出明细。